# PI-ERE End-to-End Pipeline Demo

This notebook demonstrates the complete PI-ERE pipeline from data ingestion to visualization.

## Pipeline Overview

We will demonstrate:
1. **Data Loading**: Load harmonized panel data (or create synthetic demo data)
2. **Embedding Generation**: Create time-series embeddings for similarity search
3. **Similarity Search**: Build FAISS index and find historical analogues
4. **Risk Forecasting**: Train forecaster and predict 6-month ahead risks
5. **Anomaly Detection**: Detect unusual patterns in current data
6. **Early Warning System**: Generate and prioritize operational alerts
7. **Explainability**: Generate narrative explanations for predictions
8. **Visualization Dashboard**: Create comprehensive risk visualization

**Focus Region:** Democratic Republic of Congo (COD) - copper and cobalt mining

**Note:** This notebook can run without real API data by generating synthetic demonstration data.

In [ ]:
# Imports and Setup
import sys
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# PI-ERE imports
from pi_ere.data.ingest import DataIngestionOrchestrator
from pi_ere.data.sources import CommoditiesSource
from pi_ere.data.harmonize import DataHarmonizer
from pi_ere.embeddings import TimeSeriesEmbedder, TextEmbedder
from pi_ere.models import BaselineForecaster, RiskForecaster, RiskForecast
from pi_ere.vector_db import SimilaritySearch
from pi_ere.anomaly import AnomalyDetector, EarlyWarningSystem
from pi_ere.reporting import ExplainabilityEngine, RiskVisualizer

# Plotting setup
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("PI-ERE End-to-End Pipeline Demo")
print("=" * 80)
print("Environment configured successfully!")

## Section 1: Data Loading

Load harmonized panel data from previous ingestion, or create synthetic demo data if not available.

In [ ]:
# Configuration
DEMO_REGION = 'COD'  # Democratic Republic of Congo
LOOKBACK_DAYS = 365  # 1 year of historical data
FORECAST_HORIZON = 180  # 6 months ahead

# Try to load existing harmonized data
processed_dir = Path.cwd().parent / 'data' / 'processed'
data_files = list(processed_dir.glob('*harmonized*.parquet')) if processed_dir.exists() else []

if data_files:
    print(f"Loading existing harmonized data from: {data_files[0]}")
    harmonized_df = pd.read_parquet(data_files[0])
    
    # Filter to demo region
    harmonized_df = harmonized_df[harmonized_df['region'] == DEMO_REGION].copy()
    print(f"Loaded {len(harmonized_df):,} records for {DEMO_REGION}")
    
else:
    print("No harmonized data found. Creating synthetic demo data...")
    
    # Create synthetic time-series data
    np.random.seed(42)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=LOOKBACK_DAYS)
    dates = pd.date_range(start_date, end_date, freq='D')
    
    # Create various features with realistic patterns
    features = []
    
    # Conflict events (Poisson process with trend)
    conflict_baseline = 5
    conflict_trend = np.linspace(0, 3, len(dates))
    conflict_events = np.random.poisson(conflict_baseline + conflict_trend + 
                                       2 * np.sin(np.arange(len(dates)) * 2 * np.pi / 30))
    
    # Commodity prices (random walk with volatility)
    copper_price = 8000 + np.cumsum(np.random.randn(len(dates)) * 100)
    cobalt_price = 50000 + np.cumsum(np.random.randn(len(dates)) * 500)
    
    # Economic indicators
    gdp_growth = 3.5 + np.random.randn(len(dates)) * 0.5
    inflation = 5.0 + np.cumsum(np.random.randn(len(dates)) * 0.1)
    
    # Political risk (0-100 scale)
    political_risk = 65 + 10 * np.sin(np.arange(len(dates)) * 2 * np.pi / 365) + \
                     np.random.randn(len(dates)) * 3
    
    # Create harmonized dataframe
    data_dict = {
        'acled_event_count': conflict_events,
        'acled_fatalities': conflict_events * np.random.randint(0, 5, len(dates)),
        'copper_price_usd': copper_price,
        'cobalt_price_usd': cobalt_price,
        'gdp_growth_rate': gdp_growth,
        'inflation_rate': inflation,
        'political_risk_index': political_risk,
        'gdelt_event_count': conflict_events * 2 + np.random.poisson(10, len(dates)),
        'gdelt_avg_tone': np.random.randn(len(dates)) * 2 - 1,
    }
    
    # Convert to long format
    records = []
    for feature_name, values in data_dict.items():
        for date, value in zip(dates, values):
            records.append({
                'region': DEMO_REGION,
                'date': date,
                'feature_name': feature_name,
                'value': value
            })
    
    harmonized_df = pd.DataFrame(records)
    print(f"Created {len(harmonized_df):,} synthetic records")

# Display summary
print(f"\nData Summary:")
print(f"  Region: {DEMO_REGION}")
print(f"  Features: {harmonized_df['feature_name'].nunique()}")
print(f"  Date range: {harmonized_df['date'].min().date()} to {harmonized_df['date'].max().date()}")
print(f"  Shape: {harmonized_df.shape}")

print("\nSample features:")
print(harmonized_df['feature_name'].unique())

## Section 2: Generate Embeddings

Create dense vector representations of time-series data for similarity search.

In [ ]:
# Initialize time-series embedder
print("Generating time-series embeddings...")

ts_embedder = TimeSeriesEmbedder(
    model_name='rocket',  # ROCKET: Fast time-series features
    embedding_dim=128
)

# Convert to wide format for embedding
wide_df = harmonized_df.pivot_table(
    index=['region', 'date'],
    columns='feature_name',
    values='value'
).reset_index()

# Fill missing values
feature_cols = [col for col in wide_df.columns if col not in ['region', 'date']]
wide_df[feature_cols] = wide_df[feature_cols].fillna(method='ffill').fillna(0)

print(f"Wide format shape: {wide_df.shape}")

# Extract time-series arrays
# Use a sliding window approach: create embeddings for 30-day windows
WINDOW_SIZE = 30
embeddings = []
metadata = []

for i in range(WINDOW_SIZE, len(wide_df)):
    window_data = wide_df.iloc[i-WINDOW_SIZE:i][feature_cols].values
    
    # Flatten to 1D for ROCKET
    ts_array = window_data.flatten()
    
    # Generate embedding
    embedding = ts_embedder.encode(ts_array.reshape(1, -1))
    embeddings.append(embedding[0])
    
    # Store metadata
    metadata.append({
        'region': DEMO_REGION,
        'end_date': wide_df.iloc[i]['date'],
        'window_start': wide_df.iloc[i-WINDOW_SIZE]['date'],
        'index': i
    })

embeddings = np.array(embeddings)
print(f"\nGenerated {len(embeddings)} embeddings")
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

## Section 3: Build Similarity Index

Build FAISS index for fast similarity search of historical analogues.

In [ ]:
# Initialize similarity search
print("Building similarity index...")

similarity_search = SimilaritySearch(
    embedding_dim=embeddings.shape[1],
    index_type='flat'  # Exact search (use 'ivf' for large datasets)
)

# Add embeddings to index
similarity_search.add_embeddings(
    embeddings=embeddings,
    metadata=metadata
)

print(f"Index built with {similarity_search.index.ntotal} vectors")

# Find historical analogues for most recent window
query_embedding = embeddings[-1].reshape(1, -1)
print(f"\nSearching for top 5 historical analogues...")

results = similarity_search.search(
    query_embedding=query_embedding,
    k=6  # Top 6 (includes query itself)
)

print("\nTop Historical Analogues:")
print("=" * 80)
for i, (distance, meta) in enumerate(zip(results['distances'][0][1:], 
                                          results['metadata'][1:])):
    print(f"{i+1}. Date: {meta['end_date'].date()} | "
          f"Similarity: {1 / (1 + distance):.4f} | "
          f"Window: {meta['window_start'].date()} to {meta['end_date'].date()}")

## Section 4: Train Forecaster & Generate Predictions

Train risk forecaster and predict 6-month ahead operational risks.

In [ ]:
# Initialize risk forecaster
print("Training Risk Forecaster...")

forecaster = RiskForecaster(
    model_type='prophet',  # Use Prophet for demo (faster than transformers)
    forecast_horizon=FORECAST_HORIZON
)

# Prepare training data
# Focus on key risk indicators
risk_features = ['acled_event_count', 'acled_fatalities', 'political_risk_index']

training_data = wide_df[['date'] + risk_features].copy()
training_data = training_data.set_index('date')

print(f"Training on {len(training_data)} days of data")
print(f"Features: {risk_features}")

# Fit forecaster
forecaster.fit(training_data)

print("\nGenerating forecasts...")

# Generate predictions
forecast = forecaster.predict(horizon=FORECAST_HORIZON)

print(f"\nForecast Summary:")
print(f"  Horizon: {FORECAST_HORIZON} days")
print(f"  Region: {forecast.region}")
print(f"  Risk level: {forecast.risk_level}")
print(f"  Confidence: {forecast.confidence:.2%}")
print(f"  Key drivers: {', '.join(forecast.drivers[:3])}")

# Plot forecasts
fig, axes = plt.subplots(len(risk_features), 1, figsize=(15, 10))

for i, feature in enumerate(risk_features):
    ax = axes[i] if len(risk_features) > 1 else axes
    
    # Historical data
    ax.plot(training_data.index, training_data[feature], 
            label='Historical', color='blue', alpha=0.7)
    
    # Forecast
    if feature in forecast.predictions:
        forecast_dates = pd.date_range(
            training_data.index[-1] + timedelta(days=1),
            periods=FORECAST_HORIZON,
            freq='D'
        )
        ax.plot(forecast_dates, forecast.predictions[feature], 
                label='Forecast', color='red', linestyle='--', alpha=0.7)
    
    ax.set_title(f'{feature.replace("_", " ").title()}')
    ax.set_ylabel('Value')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.show()

print("\nForecast generated successfully!")

## Section 5: Anomaly Detection

Detect unusual patterns in current data that may indicate emerging risks.

In [ ]:
# Initialize anomaly detector
print("Running Anomaly Detection...")

anomaly_detector = AnomalyDetector(
    method='isolation_forest',
    contamination=0.1  # Expect 10% anomalies
)

# Fit on historical data
print(f"Training on {len(training_data)} observations")
anomaly_detector.fit(training_data[risk_features])

# Detect anomalies in recent data (last 30 days)
recent_data = training_data[risk_features].iloc[-30:]
anomalies = anomaly_detector.detect(recent_data)

print(f"\nAnomaly Detection Results:")
print(f"  Total observations: {len(recent_data)}")
print(f"  Anomalies detected: {anomalies.sum()}")
print(f"  Anomaly rate: {anomalies.sum() / len(recent_data):.2%}")

# Get anomaly scores
scores = anomaly_detector.score(recent_data)

# Plot anomalies
fig, ax = plt.subplots(figsize=(15, 6))

# Plot conflict events with anomalies highlighted
dates = recent_data.index
values = recent_data['acled_event_count'].values

# Normal points
ax.scatter(dates[~anomalies], values[~anomalies], 
           color='blue', alpha=0.6, label='Normal', s=50)

# Anomalies
if anomalies.sum() > 0:
    ax.scatter(dates[anomalies], values[anomalies], 
               color='red', alpha=0.8, label='Anomaly', s=100, marker='X')

ax.plot(dates, values, color='gray', alpha=0.3, linestyle='-')
ax.set_title('Conflict Events - Anomaly Detection (Last 30 Days)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Event Count')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show most anomalous dates
if anomalies.sum() > 0:
    print("\nMost Anomalous Dates:")
    print("=" * 80)
    anomaly_dates = dates[anomalies]
    anomaly_scores = scores[anomalies]
    top_indices = np.argsort(anomaly_scores)[:5]
    
    for idx in top_indices:
        date = anomaly_dates[idx]
        score = anomaly_scores[idx]
        print(f"Date: {date.date()} | Anomaly Score: {score:.4f}")

## Section 6: Generate Alerts

Generate and prioritize operational alerts based on forecasts and anomalies.

In [ ]:
# Initialize early warning system
print("Generating Early Warning Alerts...")

ews = EarlyWarningSystem(
    alert_threshold=0.7,  # Alert if risk probability > 70%
    lookback_window=30
)

# Generate alerts based on forecast and anomalies
alerts = ews.generate_alerts(
    forecast=forecast,
    anomalies=anomalies,
    current_data=recent_data
)

print(f"\nGenerated {len(alerts)} alerts")

# Display prioritized alerts
print("\nPrioritized Alerts:")
print("=" * 80)

for i, alert in enumerate(alerts[:5], 1):
    print(f"\n{i}. {alert.title}")
    print(f"   Severity: {alert.severity.upper()}")
    print(f"   Probability: {alert.probability:.1%}")
    print(f"   Category: {alert.category}")
    print(f"   Region: {alert.region}")
    print(f"   Issued: {alert.timestamp.strftime('%Y-%m-%d %H:%M')}")
    print(f"   Message: {alert.message}")
    print(f"   Recommended Actions:")
    for action in alert.recommended_actions:
        print(f"     - {action}")

# Alert severity distribution
severity_counts = {}
for alert in alerts:
    severity_counts[alert.severity] = severity_counts.get(alert.severity, 0) + 1

if severity_counts:
    print("\nAlert Severity Distribution:")
    for severity, count in sorted(severity_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {severity.upper()}: {count}")

## Section 7: Explainability

Generate human-readable explanations for predictions and alerts.

In [ ]:
# Initialize explainability engine
print("Generating Explanations...")

explainer = ExplainabilityEngine(
    model=forecaster,
    method='shap'  # SHAP for feature importance
)

# Generate explanation for forecast
explanation = explainer.explain(
    forecast=forecast,
    historical_data=training_data,
    alerts=alerts[:3]  # Top 3 alerts
)

print("\n" + "=" * 80)
print("RISK FORECAST EXPLANATION")
print("=" * 80)

print(f"\nRegion: {explanation.region}")
print(f"Forecast Period: {explanation.forecast_period}")
print(f"Risk Level: {explanation.risk_level}")
print(f"Confidence: {explanation.confidence:.1%}")

print(f"\nNarrative Summary:")
print("-" * 80)
print(explanation.narrative)

print(f"\nKey Risk Drivers:")
print("-" * 80)
for i, (driver, importance) in enumerate(explanation.drivers.items(), 1):
    print(f"{i}. {driver}: {importance:.1%} contribution")

print(f"\nHistorical Context:")
print("-" * 80)
print(explanation.context)

if explanation.recommendations:
    print(f"\nRecommendations:")
    print("-" * 80)
    for i, rec in enumerate(explanation.recommendations, 1):
        print(f"{i}. {rec}")

# Feature importance visualization
if explanation.drivers:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    drivers = list(explanation.drivers.keys())
    importances = list(explanation.drivers.values())
    
    y_pos = np.arange(len(drivers))
    ax.barh(y_pos, importances, alpha=0.7, color='steelblue')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(drivers)
    ax.set_xlabel('Importance')
    ax.set_title('Feature Importance for Risk Forecast', fontsize=14)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

## Section 8: Visualization Dashboard

Create comprehensive risk visualization dashboard.

In [ ]:
# Initialize risk visualizer
print("Creating Risk Dashboard...")

visualizer = RiskVisualizer()

# Create comprehensive dashboard
dashboard = visualizer.create_region_dashboard(
    region=DEMO_REGION,
    historical_data=training_data,
    forecast=forecast,
    alerts=alerts,
    anomalies=anomalies,
    explanation=explanation
)

print("\nDashboard Components:")
print("  1. Time-series trends with forecasts")
print("  2. Risk heatmap")
print("  3. Alert timeline")
print("  4. Anomaly detection results")
print("  5. Feature importance")
print("  6. Historical analogues")

# Display dashboard
dashboard.render()

# Export dashboard
output_dir = Path.cwd().parent / 'outputs' / 'dashboards'
output_dir.mkdir(parents=True, exist_ok=True)

dashboard_path = output_dir / f'risk_dashboard_{DEMO_REGION}_{datetime.now().strftime("%Y%m%d")}.html'
dashboard.save(dashboard_path)

print(f"\nDashboard saved to: {dashboard_path}")

## Summary & Next Steps

### What We Demonstrated

This notebook walked through the complete PI-ERE pipeline:

1. **Data Loading**: Loaded harmonized panel data (or created synthetic demo data)
2. **Embedding Generation**: Created 128-dimensional time-series embeddings using ROCKET
3. **Similarity Search**: Built FAISS index and found top historical analogues
4. **Risk Forecasting**: Trained Prophet model and forecast 6-month ahead risks
5. **Anomaly Detection**: Identified unusual patterns using Isolation Forest
6. **Early Warning System**: Generated and prioritized operational alerts
7. **Explainability**: Created narrative explanations with feature importance
8. **Visualization**: Built comprehensive risk dashboard

### Key Insights

- The pipeline is modular and extensible
- Each component can be used independently or as part of the full pipeline
- The system handles missing data gracefully with synthetic fallbacks
- Explainability is integrated throughout for transparency

### Next Steps

1. **Scale to Multiple Regions**: Extend analysis to all mining regions in Africa
2. **Real-time Monitoring**: Set up automated pipeline for daily updates
3. **Model Refinement**: 
   - Test transformer models (Chronos, TimesFM) for better accuracy
   - Incorporate external event data (GDELT, news)
   - Add geospatial features
4. **Integration**: 
   - Build REST API for forecast serving
   - Create Streamlit/Dash web interface
   - Connect to alerting systems (email, Slack)
5. **Validation**:
   - Backtest on historical disruptions
   - Calculate precision/recall for alerts
   - A/B test different models

### Resources

- **Documentation**: See `docs/` directory for detailed API reference
- **Data Exploration**: `01_data_exploration.ipynb`
- **CLI Tool**: Use `pi-ere` command-line tool for batch processing
- **Configuration**: Edit `config/config.yaml` for customization

### Contact

For questions or contributions, please see the project repository.